<a href="https://www.kaggle.com/code/tarzon/telco-customer-churn?scriptVersionId=62015685" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

Telco-Customer-Churn- using ANN

In this article, I'm going to use a customer base dataset from an anonymous carrier, made available by the platform IBM Developer.

The main goal is to develop a machine learning model capable to predict customer churn based on the customer's data available. I will use mainly Python, Pandas, and Scikit-Learn libraries for this implementation. The complete code you can find on my GitHub. To accomplish that, we will go through the below steps:

Exploratory analysis
Data preparation
Train, tune and evaluate machine learning models


Problem Definition:
Based on the introduction the key challenge is to predict if an individual customer will churn or not.

Getting Started...

Importing libraries

In [ ]:
import numpy              as np
import pandas             as pd
import matplotlib.pyplot  as plt
import seaborn            as sns
import tensorflow         as tf


from sklearn.preprocessing   import OneHotEncoder
from sklearn.compose         import ColumnTransformer
from sklearn.preprocessing   import StandardScaler


from sklearn.model_selection  import train_test_split
from sklearn.linear_model     import LinearRegression
from sklearn.linear_model     import LogisticRegression
from sklearn.neighbors        import KNeighborsClassifier
from sklearn.tree             import DecisionTreeClassifier
from sklearn.ensemble         import RandomForestClassifier
from sklearn.naive_bayes      import GaussianNB
from sklearn.linear_model     import SGDClassifier
from sklearn.metrics          import confusion_matrix,accuracy_score


Load dataset

In [ ]:
#load dataset
df=pd.read_csv('../input/telcocustomerchurn/Telco-Customer-Churn.csv')

In [ ]:
# check first 5 entries
df.head()

Data Dictionary

* customerID - Custumer unique identifier
* gender - Customer gender - ['Female' 'Male']
* SeniorCitizen - Elderly or retired person, a senior citizen is someone who has at least attained the age of 60 of 65 years
* Partner - - ['No' 'Yes']
* Dependents - If customer has dependents - ['No' 'Yes']
* Tenure - Customer lifespan (in months)
* PhoneService - - ['No' 'Yes']
* MultipleLines - - ['No' 'No phone service' 'Yes']
* InternetService - - ['No' 'No internet service' 'Yes']
* OnlineSecurity - - ['No' 'No internet service' 'Yes']
* OnlineBackup - - ['No' 'No internet service' 'Yes']
* DeviceProtection - - ['No' 'No internet service' 'Yes']
* TechSupport - - ['No' 'No internet service' 'Yes']
* StreamingTV - - ['No' 'No internet service' 'Yes']
* StreamingMovies - - ['No' 'No internet service' 'Yes']
* Contract - Type of contract - ['Month-to-month' 'One year' 'Two year']
* PaperlessBilling - - ['No' 'Yes']
* PaymentMethod - payment method - ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']
* MonthlyCharges - Monthly Recurring Charges
* TotalCharges - Life time value
* Churn - Churn value, the targer vector - ['No' 'Yes']

In [ ]:
df.shape

In [ ]:
#Get a summary on the data frame
df.info()

In [ ]:
#Get statistical information on numerical features.
df.describe()

Checking missing values

In [ ]:
#number of missing values
df.isnull().sum()

In [ ]:
df.dtypes

In [ ]:
corr=df.corr()

In [ ]:
plt.figure(figsize=(25,10))
sns.heatmap(corr,vmax=0.7,square=True,annot=True)

In [ ]:
plt.figure(figsize=(8,8))
sns.countplot(x=df['Churn'],palette='Blues')
plt.tight_layout()

In [ ]:
services = ['PhoneService','MultipleLines','InternetService','OnlineSecurity',
           'OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies']

color_list=['#B0E0E6','#C1F0F6']
width=0.8
fig, axes = plt.subplots(nrows = 3,ncols = 3,figsize = (18,18))
for i, item in enumerate(services):
    if i < 3:
        ax = df[item].value_counts().plot(kind = 'bar',ax=axes[i,0],rot = 0,color=color_list,width=width)
        
    elif i >=3 and i < 6:
        ax = df[item].value_counts().plot(kind = 'bar',ax=axes[i-3,1],rot = 0,color=color_list,width=width)
        
    elif i < 9:
        ax = df[item].value_counts().plot(kind = 'bar',ax=axes[i-6,2],rot = 0,color=color_list,width=width)
    ax.set_title(item)
    
    for p in ax.patches:
        text = '{:.1f}% ({})'.format(100 * p.get_height() / df.shape[0],p.get_height())
        width = p.get_width()
        height = p.get_height() 
        x, y = p.get_xy() 
        ax.annotate(text, (x + width/2, y + height*0.5), ha='center',va='center')
       
   

In [ ]:
contract_attributes = ['Contract', 'PaperlessBilling', 'PaymentMethod']
color_list=['#B0E0E6','#C1F0F6']
width=0.8
fig, axes = plt.subplots(nrows = 3,ncols = 3,figsize = (18,18))
for i, item in enumerate(contract_attributes):
    if i < 3:
        ax = df[item].value_counts().plot(kind = 'bar',ax=axes[i,0],rot = 0,color=color_list,width=width)
        
    elif i >=3 and i < 6:
        ax = df[item].value_counts().plot(kind = 'bar',ax=axes[i-3,1],rot = 0,color=color_list,width=width)
        
    elif i < 9:
        ax = df[item].value_counts().plot(kind = 'bar',ax=axes[i-6,2],rot = 0,color=color_list,width=width)
    ax.set_title(item)
    
    for p in ax.patches:
        text = '{:.1f}% ({})'.format(100 * p.get_height() / df.shape[0],p.get_height())
        width = p.get_width()
        height = p.get_height() 
        x, y = p.get_xy() 
        ax.annotate(text, (x + width/2, y + height*0.5), ha='center',va='center')

In [ ]:
transform = ColumnTransformer([('One',OneHotEncoder(),[1,3]),('sc',StandardScaler(),[2,5])],remainder='passthrough')

In [ ]:
x=transform.fit_transform(df)


In [ ]:
column=['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport','StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

In [ ]:
from sklearn.preprocessing import LabelEncoder
cols = ('customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport','StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn')
# Process columns and apply LabelEncoder to categorical features
for c in cols:
    lbl = LabelEncoder() 
    lbl.fit(list(df[c].values)) 
    df[c] = lbl.transform(list(df[c].values))

# Check shape        
print('Shape data: {}'.format(df.shape))

In [ ]:
x=df.drop('Churn',axis=1)
y=df['Churn']


In [ ]:
x.head()

In [ ]:
y

In [ ]:
from sklearn.model_selection import train_test_split
xtrain, xtest, ytrain, ytest = train_test_split(x,y,test_size=0.2, random_state = 42)

In [ ]:
model=LinearRegression()
model.fit(xtrain,ytrain)
ypred=model.predict(xtest)
acc_lin_reg=round(model.score(xtrain,ytrain)*100,2 )
print(str(acc_lin_reg)+ ' percent')

In [ ]:
model = LogisticRegression(random_state=10)
model.fit(xtrain,ytrain)
ypred=model.predict(xtest)
print(model.score(xtrain,ytrain))

acc_log_reg=round(model.score(xtrain,ytrain)*100,2 )
print(str(acc_log_reg)+' percent')

In [ ]:
model = KNeighborsClassifier(n_neighbors = 3)
model.fit(xtrain, ytrain)
ypred = model.predict(xtest)
acc_knn = round(model.score(xtrain, ytrain) * 100, 2)
print (acc_knn)

In [ ]:
model = DecisionTreeClassifier(max_depth = 15,random_state=100)
model.fit(xtrain, ytrain)
y_pred = model.predict(xtest)
acc_decision_tree = round(model.score(xtrain, ytrain) * 100, 2)
print (acc_decision_tree)


In [ ]:
model = RandomForestClassifier(n_estimators=100,max_depth = 15)
model.fit(xtrain, ytrain)
ypred = model.predict(xtest)
acc_random_forest = round(model.score(xtrain, ytrain) * 100, 2)
print (acc_random_forest)

In [ ]:
model = GaussianNB()
model.fit(xtrain, ytrain)
ypred = model.predict(xtest)
acc_gnb = round(model.score(xtrain, ytrain) * 100, 2)
print (acc_gnb)

In [ ]:
model = SGDClassifier()
model.fit(xtrain, ytrain)
ypred = model.predict(xtest)
acc_sgd = round(model.score(xtrain, ytrain) * 100, 2)
print (acc_sgd)

In [ ]:
models = pd.DataFrame({
    'Model': ['Linear Regression','Logistic Regression', 
              'KNN', 'Decision Tree', 'Random Forest', 'Naive Bayes', 
              'Stochastic Gradient Decent'],
    
    'Score': [acc_lin_reg,acc_log_reg,acc_knn,  
              acc_decision_tree, acc_random_forest, acc_gnb, 
             acc_sgd]
    })

models.sort_values(by='Score', ascending=False)

In [ ]:
model = RandomForestClassifier(n_estimators=100,max_depth = 15)
model.fit(xtrain, ytrain)
ypred = model.predict(xtest)
acc_random_forest = round(model.score(xtrain, ytrain) * 100, 2)
print (acc_random_forest)

In [ ]:
ypred=model.predict(xtest)
print(confusion_matrix(ytest,ypred))

In [ ]:
print(accuracy_score(ytest,ypred))

Artificial Neural **Network**

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LeakyReLU,PReLU,ELU
from keras.layers import Dropout


# Initialising the ANN
classifier =  tf.keras.models.Sequential()

# Adding the hidden layer
classifier.add(tf.keras.layers.Dense(units=6, activation='relu'))

# Adding the second hidden layer
classifier.add(tf.keras.layers.Dense(units=6, activation='relu'))

# Adding the thred hidden layer
classifier.add(tf.keras.layers.Dense(units=6, activation='relu'))

# Adding the output layer
classifier.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))



In [ ]:
# Compiling the ANN
classifier.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Fitting the ANN to the Training set
classifier.fit(xtrain, ytrain, batch_size=32, epochs=100)